# Basket Economics: Cross-Platform Shopping Cost Optimisation

**Portfolio demonstration — all included offer data is synthetic.**

This notebook explores platform offers, calculates final basket costs, and compares single-platform and split-basket purchasing strategies.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from generate_data import build_data
from optimizer import BasketItem, optimise_basket, compare_strategies

## 2. Load and inspect offers

The included data contains 15 fictional products across three fictional shopping platforms.

In [ ]:
offers_path = ROOT / 'data' / 'offers.csv'
offers = pd.read_csv(offers_path) if offers_path.exists() else build_data()
print('Rows and columns:', offers.shape)
offers.head()

In [ ]:
print('Missing cells:', offers.isna().sum().sum())
print('Duplicate product-platform pairs:', offers.duplicated(['product_id', 'platform']).sum())
display(offers.groupby('platform').agg(offers=('product_id','count'), stock_rate=('in_stock','mean'), avg_discount=('discount_pct','mean')).round(3))

## 3. Offer landscape

In [ ]:
platform_summary = offers.groupby('platform', as_index=False).agg(avg_listed_price=('listed_price','mean'), avg_discount=('discount_pct','mean'), stock_rate=('in_stock','mean'))
platform_summary

In [ ]:
ax = platform_summary.plot(x='platform', y='avg_listed_price', kind='bar', legend=False, title='Average listed price by platform — synthetic data')
ax.set_ylabel('INR'); plt.tight_layout()

## 4. Define the demo basket

One wireless mouse, one USB-C hub, two water bottles, and one backpack.

In [ ]:
basket = [BasketItem('P001', 1), BasketItem('P003', 1), BasketItem('P005', 2), BasketItem('P008', 1)]
[(item.product_id, item.quantity) for item in basket]

## 5. Optimise the allocation

In [ ]:
allocation, summary = optimise_basket(offers, basket)
allocation

In [ ]:
summary

## 6. Compare purchasing strategies

In [ ]:
strategy = compare_strategies(offers, basket)
strategy

In [ ]:
ax = strategy.sort_values('total_cost').plot(x='strategy', y='total_cost', kind='bar', legend=False, title='Basket cost by strategy')
ax.set_ylabel('INR'); ax.tick_params(axis='x', rotation=15); plt.tight_layout()

## 7. Interpret results

Compare the total after discounts, eligible coupons, and platform-level shipping. All generated values are synthetic and the illustrative difference is not observed marketplace savings.

In [ ]:
best_single = strategy[strategy.strategy.str.startswith('All items')].total_cost.min()
split_cost = strategy.loc[strategy.strategy == 'Optimised split basket', 'total_cost'].iloc[0]
print(f'Best single-platform cost: INR {best_single:,.2f}')
print(f'Optimised split-basket cost: INR {split_cost:,.2f}')
print(f'Illustrative difference: INR {best_single-split_cost:,.2f}')

## 8. Export results

In [ ]:
allocation.to_csv(ROOT / 'outputs' / 'optimised_allocation.csv', index=False)
strategy.to_csv(ROOT / 'outputs' / 'strategy_comparison.csv', index=False)
print('Saved core result tables.')